In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_models, get_features, ModelTypes, model_names #, get_features
from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
from dinosaw.utils import do_2D_pca
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [2]:
selected_model = 'alibi_dv2_coco'
selected_models: tuple[ModelTypes, ...] = ('dv2', 'alibi_dv2_coco', 'dv3', 'alibi_dv3')

models = get_models(selected_models, '../../trained_models', device=DEVICE,  conf_path='../../dinov3')
S = models[selected_models[0]].stride

n_dims = 384

In [3]:
from skimage.exposure import equalize_adapthist, equalize_hist
image_names = ['labradors', 'new_york', 'bimodal', 'biphase_steel_crop']

shortest_side = 683
longest_side = 1024

images = []

for image_name in image_names:
    image = Image.open(f'../paper_figures/data/more_pcas/{image_name}.png')
    ih, iw = image.size[::-1]
    scale = shortest_side / min(ih, iw)
    image = image.resize((int(iw * scale), int(ih * scale)))
    
    ox, oy = (image.size[0] - longest_side) // 2, (image.size[1] - shortest_side) // 2
    image = image.crop((ox, oy, ox + longest_side, oy + shortest_side))

    if image_name in ("bimodal", "biphase_steel_crop"):
        image_arr = np.array(image)
        image_arr = equalize_adapthist(image_arr, clip_limit=0.008)
        image = Image.fromarray((image_arr * 255).astype(np.uint8))

    images.append(image)


In [4]:
features: dict[ModelTypes, list[np.ndarray]] = {key: [] for key in selected_models}

for key, model in models.items():
    for image in images:
        feats = get_features(model, image, device=DEVICE)
        reduced = do_2D_pca(feats, 9, pre_norm='std',  post_norm='minmax')[:, :, 0:3]
        features[key].append(reduced)

In [5]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [6]:
# %%capture
plt.style.use("thesis.mplstyle")

n_rows, n_cols = len(selected_models) + 1, len(images)
W, H = 7, 2.3 * (2.2)

titles = ["Labradors", "New York", "Bimodal cathode", "Biphase steel"]
fig, axs = plt.subplots(n_rows, n_cols, figsize=(W, H))

for col, image in enumerate(images):
    axs[0, col].imshow(image, rasterized=True)
    hide_axes(axs[0, col])
    axs[0, col].set_title(titles[col])

    for row, key in enumerate(selected_models):
        ax = axs[row + 1, col]
        ax.imshow(features[key][col], rasterized=True)
        hide_axes(ax)

        if col == 0:
            weight = 700 if 'alibi' in key else 500
            name = model_names[key]
            name = rf"\textbf{{{name}}}" if 'alibi' in name.lower() else name
            name = name.replace('(COCO)', '')
            ax.set_ylabel(name)

# plt.tight_layout()
plt.savefig('out/pcas.pdf', dpi=300)
plt.close()